# OLA Ride Insights | Data Preprocessing

**Goal:** take the raw `OLA_DataSet.xlsx` (July 2024 bookings), audit it, clean it, add a few useful columns, and export `ola_data_clean.csv` for the SQL, Python and dashboard stages.

**Steps**
1. Load the data and take a first look
2. Audit data quality (nulls, duplicates, formats, categories)
3. Clean and standardise
4. Feature engineering
5. Validate and export

Run it from the `notebooks/` folder. It reads `../data/raw/OLA_DataSet.xlsx` and writes `../data/ola_data_clean.csv`.

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_PATH = "../data/raw/OLA_DataSet.xlsx"
OUTPUT_PATH = "../data/ola_data_clean.csv"
PARQUET_PATH = "../data/ola_data_clean.parquet"   # compact copy (about 3 MB) used by the Streamlit app

# The workbook has 3 sheets. "July" holds the 103k bookings, the other two are
# small summary/lookup tabs (average distance chart data and vehicle icon links).
print(pd.ExcelFile(DATA_PATH).sheet_names)

raw = pd.read_excel(DATA_PATH, sheet_name="July")
df = raw.copy()          # keep the raw frame untouched for before/after comparison
df.shape

['July', 'Sheet1', 'Sheet2']


(103024, 20)

In [2]:
df.head()

,Date,Time,Booking_ID,Booking_Status,Customer_ID,Vehicle_Type,Pickup_Location,Drop_Location,V_TAT,C_TAT,Canceled_Rides_by_Customer,Canceled_Rides_by_Driver,Incomplete_Rides,Incomplete_Rides_Reason,Booking_Value,Payment_Method,Ride_Distance,Driver_Ratings,Customer_Rating,Vehicle Images
0,2024-07-26 14:00:00,14:00:00,CNR7153255142,Canceled by Driver,CID713523,Prime Sedan,Tumkur Road,RT Nagar,NaN,NaN,NaN,Personal & Car related issue,NaN,NaN,444,NaN,0,NaN,NaN,https://cdn-icons-png.flaticon.com/128/14183/1...
1,2024-07-25 22:20:00,22:20:00,CNR2940424040,Success,CID225428,Bike,Magadi Road,Varthur,203.0,30.0,NaN,NaN,No,NaN,158,Cash,13,4.1,4.0,https://cdn-icons-png.flaticon.com/128/9983/99...
2,2024-07-30 19:59:00,19:59:00,CNR2982357879,Success,CID270156,Prime SUV,Sahakar Nagar,Varthur,238.0,130.0,NaN,NaN,No,NaN,386,UPI,40,4.2,4.8,https://cdn-icons-png.flaticon.com/128/9983/99...
3,2024-07-22 03:15:00,03:15:00,CNR2395710036,Canceled by Customer,CID581320,eBike,HSR Layout,Vijayanagar,NaN,NaN,Driver is not moving towards pickup location,NaN,NaN,NaN,384,NaN,0,NaN,NaN,https://cdn-icons-png.flaticon.com/128/6839/68...
4,2024-07-02 09:02:00,09:02:00,CNR1797421769,Success,CID939555,Mini,Rajajinagar,Chamarajpet,252.0,80.0,NaN,NaN,No,NaN,822,Credit Card,45,4.0,3.0,https://cdn-icons-png.flaticon.com/128/3202/32...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103024 entries, 0 to 103023
Data columns (total 20 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   Date                        103024 non-null  datetime64[us]
 1   Time                        103024 non-null  object        
 2   Booking_ID                  103024 non-null  str           
 3   Booking_Status              103024 non-null  str           
 4   Customer_ID                 103024 non-null  str           
 5   Vehicle_Type                103024 non-null  str           
 6   Pickup_Location             103024 non-null  str           
 7   Drop_Location               103024 non-null  str           
 8   V_TAT                       63967 non-null   float64       
 9   C_TAT                       63967 non-null   float64       
 10  Canceled_Rides_by_Customer  10499 non-null   str           
 11  Canceled_Rides_by_Driver    18434 non-null   str  

## 2. Data quality audit

### 2.1 Missing values

In [4]:
missing = df.isna().sum().to_frame("missing")
missing["pct"] = (missing["missing"] / len(df) * 100).round(2)
missing[missing["missing"] > 0]

,missing,pct
V_TAT,39057,37.91
C_TAT,39057,37.91
Canceled_Rides_by_Customer,92525,89.81
Canceled_Rides_by_Driver,84590,82.11
Incomplete_Rides,39057,37.91
Incomplete_Rides_Reason,99098,96.19
Payment_Method,39057,37.91
Driver_Ratings,39057,37.91
Customer_Rating,39057,37.91


A lot of columns are missing for 30 to 90 percent of rows, which looks alarming at first. Before touching anything, let's check whether these gaps are **real data problems or just fields that don't apply** to a given booking (for example, a cancelled ride can't have a rating).

In [5]:
# Which columns are filled for each booking status?
filled = df.groupby("Booking_Status")[
    ["V_TAT", "C_TAT", "Canceled_Rides_by_Customer", "Canceled_Rides_by_Driver",
     "Incomplete_Rides", "Payment_Method", "Driver_Ratings", "Customer_Rating"]
].count()
filled["total_bookings"] = df["Booking_Status"].value_counts()
filled

,V_TAT,C_TAT,Canceled_Rides_by_Customer,Canceled_Rides_by_Driver,Incomplete_Rides,Payment_Method,Driver_Ratings,Customer_Rating,total_bookings
Booking_Status,,,,,,,,,
Canceled by Customer,0,0,10499,0,0,0,0,0,10499
Canceled by Driver,0,0,0,18434,0,0,0,0,18434
Driver Not Found,0,0,0,0,0,0,0,0,10124
Success,63967,63967,0,0,63967,63967,63967,63967,63967


**Takeaway:** the gaps follow the booking status exactly.

- Turnaround times, payment method, ratings and the incomplete flag exist **only** for `Success` bookings.
- A customer cancellation reason exists only for `Canceled by Customer`, a driver reason only for `Canceled by Driver`.
- `Driver Not Found` bookings have none of these.

So these are structural nulls, not errors. Dropping or mean-filling them would be wrong, and I handle them separately in the cleaning section.

### 2.2 Duplicates

In [6]:
print("Fully duplicated rows :", df.duplicated().sum())
print("Duplicate Booking_IDs :", df["Booking_ID"].duplicated().sum())

Fully duplicated rows : 0
Duplicate Booking_IDs : 0


### 2.3 Text formatting (whitespace and ID patterns)

In [7]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
ws_issues = {c: int((df[c].dropna().astype(str) != df[c].dropna().astype(str).str.strip()).sum())
             for c in text_cols}
print("Columns with leading/trailing spaces:", {k: v for k, v in ws_issues.items() if v} or "none")

print("Booking_ID format OK :", df["Booking_ID"].str.match(r"^CNR\d+$").all())
print("Customer_ID format OK:", df["Customer_ID"].str.match(r"^CID\d+$").all())

Columns with leading/trailing spaces: none
Booking_ID format OK : True
Customer_ID format OK: True


### 2.4 Categorical columns

In [8]:
for col in ["Booking_Status", "Vehicle_Type", "Payment_Method", "Incomplete_Rides",
            "Incomplete_Rides_Reason", "Canceled_Rides_by_Customer", "Canceled_Rides_by_Driver"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string(), "\n")

--- Booking_Status ---
Booking_Status
Success                 63967
Canceled by Driver      18434
Canceled by Customer    10499
Driver Not Found        10124 

--- Vehicle_Type ---
Vehicle_Type
Prime Sedan    14877
eBike          14816
Auto           14755
Prime Plus     14707
Bike           14662
Prime SUV      14655
Mini           14552 

--- Payment_Method ---
Payment_Method
NaN            39057
Cash           35022
UPI            25881
Credit Card     2435
Debit Card       629 

--- Incomplete_Rides ---
Incomplete_Rides
No     60041
NaN    39057
Yes     3926 

--- Incomplete_Rides_Reason ---
Incomplete_Rides_Reason
NaN                  99098
Customer Demand       1601
Vehicle Breakdown     1591
Other Issue            734 

--- Canceled_Rides_by_Customer ---
Canceled_Rides_by_Customer
NaN                                             92525
Driver is not moving towards pickup location     3175
Driver asked to cancel                           2670
Change of plans                        

### 2.5 Numeric columns and logical checks

In [9]:
df[["V_TAT", "C_TAT", "Booking_Value", "Ride_Distance", "Driver_Ratings", "Customer_Rating"]].describe().round(2)

,V_TAT,C_TAT,Booking_Value,Ride_Distance,Driver_Ratings,Customer_Rating
count,63967.00,63967.00,103024.00,103024.00,63967.00,63967.00
mean,170.88,84.87,548.75,14.19,4.00,4.00
std,80.80,36.01,536.54,15.78,0.58,0.58
min,35.00,25.00,100.00,0.00,3.00,3.00
25%,98.00,55.00,242.00,0.00,3.50,3.50
50%,168.00,85.00,386.00,8.00,4.00,4.00
75%,238.00,115.00,621.00,26.00,4.50,4.50
max,308.00,145.00,2999.00,49.00,5.00,5.00


In [10]:
# Ride_Distance by status: cancelled bookings show 0 km
df.groupby("Booking_Status")["Ride_Distance"].agg(["min", "max", "mean"]).round(2)

,min,max,mean
Booking_Status,,,
Canceled by Customer,0,0,0.00
Canceled by Driver,0,0,0.00
Driver Not Found,0,0,0.00
Success,1,49,22.85


In [11]:
# Date already carries the clock time, so is the separate Time column redundant?
print("Date and Time disagree on:", (df["Date"].dt.time != df["Time"]).sum(), "rows")

# Pickup and drop-off in the same place
same_loc = (df["Pickup_Location"] == df["Drop_Location"]).sum()
print(f"Pickup == Drop location: {same_loc:,} rows ({same_loc/len(df):.1%})")

# Incomplete rides sit inside the Success status
print()
print(pd.crosstab(df["Booking_Status"], df["Incomplete_Rides"].fillna("NaN")))

Date and Time disagree on: 0 rows
Pickup == Drop location: 2,121 rows (2.1%)

Incomplete_Rides        NaN     No   Yes
Booking_Status                          
Canceled by Customer  10499      0     0
Canceled by Driver    18434      0     0
Driver Not Found      10124      0     0
Success                   0  60041  3926


**Audit findings**

| Finding | What I do about it |
|---|---|
| No duplicates, no stray whitespace, IDs are consistent | Nothing to fix |
| Nulls are structural (depend on booking status) | Keep numeric nulls, label categorical ones `Not Applicable` |
| `Time` repeats what is already inside `Date` | Build one proper `booking_datetime`, keep `date` and `time` |
| `Ride_Distance` is 0 for rides that never happened | Convert to null so averages are not dragged down |
| `Incomplete_Rides = Yes` rows are all inside `Success` | Add a clear `ride_outcome` column |
| About 2% of rides have the same pickup and drop location | Flag them, but keep them (see note below) |
| `Vehicle Images` is just icon URLs | Drop |

## 3. Cleaning

### 3.1 Drop the image column and standardise column names

In [12]:
df = df.drop(columns=["Vehicle Images"])

df = df.rename(columns={
    "Canceled_Rides_by_Customer": "canceled_by_customer_reason",
    "Canceled_Rides_by_Driver":   "canceled_by_driver_reason",
    "Incomplete_Rides_Reason":    "incomplete_reason",
})
df.columns = df.columns.str.strip().str.lower()
df.columns.tolist()

['date',
 'time',
 'booking_id',
 'booking_status',
 'customer_id',
 'vehicle_type',
 'pickup_location',
 'drop_location',
 'v_tat',
 'c_tat',
 'canceled_by_customer_reason',
 'canceled_by_driver_reason',
 'incomplete_rides',
 'incomplete_reason',
 'booking_value',
 'payment_method',
 'ride_distance',
 'driver_ratings',
 'customer_rating']

### 3.2 Fix date and time

In [13]:
# 'Date' already holds the full timestamp, so it becomes booking_datetime.
df["booking_datetime"] = pd.to_datetime(df["date"])
df["date"] = df["booking_datetime"].dt.date
df["time"] = df["booking_datetime"].dt.time

df[["booking_datetime", "date", "time"]].head()

,booking_datetime,date,time
0,2024-07-26 14:00:00,2024-07-26,14:00:00
1,2024-07-25 22:20:00,2024-07-25,22:20:00
2,2024-07-30 19:59:00,2024-07-30,19:59:00
3,2024-07-22 03:15:00,2024-07-22,03:15:00
4,2024-07-02 09:02:00,2024-07-02,09:02:00


In [14]:
print("Date range:", df["booking_datetime"].min(), "to", df["booking_datetime"].max())

Date range: 2024-07-01 00:00:00 to 2024-07-31 23:58:00


### 3.3 Handle structural missing values

In [15]:
# Categorical fields that don't apply to a booking get an explicit label,
# so they are easy to filter and don't silently disappear in group-bys.
not_applicable_cols = [
    "canceled_by_customer_reason", "canceled_by_driver_reason",
    "incomplete_rides", "incomplete_reason", "payment_method",
]
df[not_applicable_cols] = df[not_applicable_cols].fillna("Not Applicable")

# One more nuance: a successful ride that was NOT incomplete has no incomplete reason,
# which is genuinely "not applicable" too, so the label is consistent.

# Numeric fields (tat, ratings) stay null on purpose.
# Filling them with 0 or the mean would invent data and distort averages.
df.isna().sum()[lambda s: s > 0]

v_tat              39057
c_tat              39057
driver_ratings     39057
customer_rating    39057
dtype: int64

### 3.4 Ride distance for rides that never happened

In [16]:
# Distance is stored as 0 for cancelled / driver-not-found bookings.
# That is "no ride", not "a 0 km ride", and it pulls averages down heavily.
never_ran = df["booking_status"] != "Success"
print("Average distance BEFORE:", round(df["ride_distance"].mean(), 2), "km")

df["ride_distance"] = df["ride_distance"].astype("float").mask(never_ran)

print("Average distance AFTER :", round(df["ride_distance"].mean(), 2), "km")

Average distance BEFORE: 14.19 km
Average distance AFTER : 22.85 km


`booking_value` is also filled for cancelled bookings (it looks like the quoted fare). I left it as it is, but keep in mind that **revenue must be calculated on completed rides only**. The `ride_outcome` column below makes that easy.

In [17]:
# Does the fare actually move with distance? (rides that ran only)
ran = df[~never_ran]
print("Correlation (booking_value vs ride_distance):", round(ran["booking_value"].corr(ran["ride_distance"]), 3))
ran.groupby(pd.cut(ran["ride_distance"], [0, 5, 10, 20, 35, 50]), observed=True)["booking_value"].median()

Correlation (booking_value vs ride_distance): 0.002


ride_distance
(0, 5]      388.0
(5, 10]     386.0
(10, 20]    387.0
(20, 35]    383.0
(35, 50]    385.0
Name: booking_value, dtype: float64

**Worth knowing:** fare is essentially unrelated to distance in this dataset (correlation near zero, and the median fare is about the same for a 3 km ride and a 45 km ride). That is not how real fares work, so I deliberately did **not** create a fare per km column, because it would just be noise. If your analysis talks about pricing, treat `booking_value` as its own variable and mention this limitation.

### 3.5 Set proper data types

In [18]:
category_cols = [
    "booking_status", "vehicle_type", "payment_method", "incomplete_rides",
    "incomplete_reason", "canceled_by_customer_reason", "canceled_by_driver_reason",
]
df[category_cols] = df[category_cols].astype("category")

df["v_tat"] = df["v_tat"].astype("float")
df["c_tat"] = df["c_tat"].astype("float")
df["booking_value"] = df["booking_value"].astype("int")
df.dtypes

date                                   object
time                                   object
booking_id                                str
booking_status                       category
customer_id                               str
vehicle_type                         category
pickup_location                           str
drop_location                             str
v_tat                                 float64
c_tat                                 float64
canceled_by_customer_reason          category
canceled_by_driver_reason            category
incomplete_rides                     category
incomplete_reason                    category
booking_value                           int64
payment_method                       category
ride_distance                         float64
driver_ratings                        float64
customer_rating                       float64
booking_datetime               datetime64[us]
dtype: object

## 4. Feature engineering

In [19]:
dt = df["booking_datetime"].dt

# Time based features
df["hour"] = dt.hour
df["day_name"] = dt.day_name()
df["day_of_week"] = dt.dayofweek + 1          # 1 = Monday ... 7 = Sunday
df["week_of_month"] = (dt.day - 1) // 7 + 1
df["is_weekend"] = dt.dayofweek >= 5

df["time_of_day"] = pd.cut(
    df["hour"],
    bins=[-1, 4, 11, 16, 20, 23],
    labels=["Late Night", "Morning", "Afternoon", "Evening", "Night"],
)

# One clear outcome column (Success contains the incomplete rides, so split them out)
df["ride_outcome"] = np.select(
    [
        (df["booking_status"] == "Success") & (df["incomplete_rides"] == "No"),
        (df["booking_status"] == "Success") & (df["incomplete_rides"] == "Yes"),
    ],
    ["Completed", "Incomplete"],
    default=df["booking_status"].astype(str),
)

# A single cancellation reason column, plus who cancelled
df["cancellation_reason"] = np.select(
    [
        df["canceled_by_customer_reason"] != "Not Applicable",
        df["canceled_by_driver_reason"] != "Not Applicable",
    ],
    [df["canceled_by_customer_reason"].astype(str), df["canceled_by_driver_reason"].astype(str)],
    default="Not Applicable",
)

# Route and a flag for rides where pickup and drop are the same place
df["route"] = df["pickup_location"] + " to " + df["drop_location"]
df["same_pickup_drop"] = df["pickup_location"] == df["drop_location"]

df["ride_outcome"].value_counts()

ride_outcome
Completed               60041
Canceled by Driver      18434
Canceled by Customer    10499
Driver Not Found        10124
Incomplete               3926
Name: count, dtype: int64

In [20]:
df[["booking_datetime", "hour", "time_of_day", "day_name", "is_weekend",
    "ride_outcome", "cancellation_reason", "route"]].head(8)

,booking_datetime,hour,time_of_day,day_name,is_weekend,ride_outcome,cancellation_reason,route
0,2024-07-26 14:00:00,14,Afternoon,Friday,False,Canceled by Driver,Personal & Car related issue,Tumkur Road to RT Nagar
1,2024-07-25 22:20:00,22,Night,Thursday,False,Completed,Not Applicable,Magadi Road to Varthur
2,2024-07-30 19:59:00,19,Evening,Tuesday,False,Completed,Not Applicable,Sahakar Nagar to Varthur
3,2024-07-22 03:15:00,3,Late Night,Monday,False,Canceled by Customer,Driver is not moving towards pickup location,HSR Layout to Vijayanagar
4,2024-07-02 09:02:00,9,Morning,Tuesday,False,Completed,Not Applicable,Rajajinagar to Chamarajpet
5,2024-07-13 04:42:00,4,Late Night,Saturday,True,Completed,Not Applicable,Kadugodi to Vijayanagar
6,2024-07-23 09:51:00,9,Morning,Tuesday,False,Completed,Not Applicable,Tumkur Road to Whitefield
7,2024-07-11 11:12:00,11,Morning,Thursday,False,Canceled by Driver,Personal & Car related issue,Bannerghatta Road to Sarjapur Road


**Note on same pickup and drop rows:** about 2% of bookings start and end in the same place. In real life that could be a round trip or a data glitch, and the dataset gives no way to tell. I flagged them with `same_pickup_drop` instead of deleting them, so you can exclude them in a specific analysis if you want. It is a small share, so it will not move the headline numbers.

## 5. Validation and export

In [21]:
# Sanity checks: the notebook stops here if anything is off
assert len(df) == len(raw) == 103024, "row count changed"
assert df["booking_id"].is_unique, "duplicate booking ids"
assert df["booking_status"].nunique() == 4

completed = df["ride_outcome"] == "Completed"
assert df.loc[~df["booking_status"].eq("Success"), "ride_distance"].isna().all()
assert df.loc[df["booking_status"].eq("Success"), "ride_distance"].notna().all()
assert df.loc[df["booking_status"].eq("Success"), ["driver_ratings", "customer_rating"]].notna().all().all()
assert df["driver_ratings"].dropna().between(1, 5).all()
assert df["customer_rating"].dropna().between(1, 5).all()
assert (df["booking_value"] > 0).all()
assert df["ride_outcome"].value_counts().sum() == len(df)

print("All checks passed")

All checks passed


In [22]:
# Before vs after
summary = pd.DataFrame({
    "raw":   [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum())],
    "clean": [df.shape[0],  df.shape[1],  int(df.isna().sum().sum())],
}, index=["rows", "columns", "total null cells"])
summary

,raw,clean
rows,103024,103024
columns,20,30
total null cells,510555,195285


In [23]:
# Final column order
ordered = [
    "booking_id", "booking_datetime", "date", "time", "hour", "day_name", "day_of_week",
    "week_of_month", "is_weekend", "time_of_day",
    "customer_id", "vehicle_type", "pickup_location", "drop_location", "route", "same_pickup_drop",
    "booking_status", "ride_outcome", "canceled_by_customer_reason", "canceled_by_driver_reason",
    "cancellation_reason", "incomplete_rides", "incomplete_reason",
    "v_tat", "c_tat", "booking_value", "ride_distance",
    "payment_method", "driver_ratings", "customer_rating",
]
df_clean = df[ordered]

df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH}: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")

# The CSV is about 29 MB, which is too big for a normal GitHub upload (25 MB limit in the browser).
# Parquet holds the same data in about 3 MB, so the Streamlit app reads this file instead.
# It needs pyarrow: pip install pyarrow
parquet_df = df_clean.copy()
for col in parquet_df.select_dtypes("category").columns:
    parquet_df[col] = parquet_df[col].astype(str)
parquet_df.to_parquet(PARQUET_PATH, index=False, compression="zstd")
print(f"Saved {PARQUET_PATH}")

Saved ../data/ola_data_clean.csv: 103,024 rows x 30 columns


Saved ../data/ola_data_clean.parquet


In [24]:
# Reload the exported file to confirm it reads back correctly
check = pd.read_csv(OUTPUT_PATH)
print(check.shape)
check.head()

(103024, 30)


,booking_id,booking_datetime,date,time,hour,day_name,day_of_week,week_of_month,is_weekend,time_of_day,customer_id,vehicle_type,pickup_location,drop_location,route,same_pickup_drop,booking_status,ride_outcome,canceled_by_customer_reason,canceled_by_driver_reason,cancellation_reason,incomplete_rides,incomplete_reason,v_tat,c_tat,booking_value,ride_distance,payment_method,driver_ratings,customer_rating
0,CNR7153255142,2024-07-26 14:00:00,2024-07-26,14:00:00,14,Friday,5,4,False,Afternoon,CID713523,Prime Sedan,Tumkur Road,RT Nagar,Tumkur Road to RT Nagar,False,Canceled by Driver,Canceled by Driver,Not Applicable,Personal & Car related issue,Personal & Car related issue,Not Applicable,Not Applicable,NaN,NaN,444,NaN,Not Applicable,NaN,NaN
1,CNR2940424040,2024-07-25 22:20:00,2024-07-25,22:20:00,22,Thursday,4,4,False,Night,CID225428,Bike,Magadi Road,Varthur,Magadi Road to Varthur,False,Success,Completed,Not Applicable,Not Applicable,Not Applicable,No,Not Applicable,203.0,30.0,158,13.0,Cash,4.1,4.0
2,CNR2982357879,2024-07-30 19:59:00,2024-07-30,19:59:00,19,Tuesday,2,5,False,Evening,CID270156,Prime SUV,Sahakar Nagar,Varthur,Sahakar Nagar to Varthur,False,Success,Completed,Not Applicable,Not Applicable,Not Applicable,No,Not Applicable,238.0,130.0,386,40.0,UPI,4.2,4.8
3,CNR2395710036,2024-07-22 03:15:00,2024-07-22,03:15:00,3,Monday,1,4,False,Late Night,CID581320,eBike,HSR Layout,Vijayanagar,HSR Layout to Vijayanagar,False,Canceled by Customer,Canceled by Customer,Driver is not moving towards pickup location,Not Applicable,Driver is not moving towards pickup location,Not Applicable,Not Applicable,NaN,NaN,384,NaN,Not Applicable,NaN,NaN
4,CNR1797421769,2024-07-02 09:02:00,2024-07-02,09:02:00,9,Tuesday,2,1,False,Morning,CID939555,Mini,Rajajinagar,Chamarajpet,Rajajinagar to Chamarajpet,False,Success,Completed,Not Applicable,Not Applicable,Not Applicable,No,Not Applicable,252.0,80.0,822,45.0,Credit Card,4.0,3.0


## Summary of what was done

- Kept all **103,024 rows** (nothing was dropped, there were no duplicates or bad IDs)
- Removed the `Vehicle Images` column and standardised column names to snake_case
- Rebuilt date and time into a proper `booking_datetime`
- Treated nulls as structural: categorical ones labelled `Not Applicable`, numeric ones left null
- Converted `ride_distance` from 0 to null for rides that never happened
- Set correct data types
- Added: `hour`, `day_name`, `day_of_week`, `week_of_month`, `is_weekend`, `time_of_day`, `ride_outcome`, `cancellation_reason`, `route`, `same_pickup_drop`
- Exported to `ola_data_clean.csv` and to a compact `ola_data_clean.parquet` (same data, about 3 MB, used by the Streamlit app)